In [ ]:
!pip install opencv-python-headless pillow numpy matplotlib

import os
import cv2
import numpy as np
from PIL import Image, ImageEnhance
import random
import zipfile
import shutil
from google.colab import files, drive
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import io

In [ ]:
print("Upload your jewelry image folders as ZIP files")
print("You can upload up to 4 ZIP files, each containing one category of jewelry images")
uploaded = files.upload()

# Extract uploaded ZIP files
input_folders = []
for filename, data in uploaded.items():
    if filename.endswith('.zip'):
        # Save the uploaded file
        with open(filename, 'wb') as f:
            f.write(data)

        # Extract the ZIP file
        folder_name = os.path.splitext(filename)[0]
        extract_path = f'/content/{folder_name}'

        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall(extract_path)

        input_folders.append(extract_path)
        print(f"Extracted {filename} to {extract_path}")

print(f"\nFound {len(input_folders)} folders to process")

Upload your jewelry image folders as ZIP files
You can upload up to 4 ZIP files, each containing one category of jewelry images


Saving necklace_images.zip to necklace_images.zip
Saving earring_best.zip to earring_best.zip
Saving necklace.zip to necklace.zip
Saving naveen_images-20250924T101516Z-1-001.zip to naveen_images-20250924T101516Z-1-001.zip
Extracted necklace_images.zip to /content/necklace_images
Extracted earring_best.zip to /content/earring_best
Extracted necklace.zip to /content/necklace
Extracted naveen_images-20250924T101516Z-1-001.zip to /content/naveen_images-20250924T101516Z-1-001

Found 4 folders to process


In [ ]:
class JewelryImageAugmentor:
    def __init__(self, input_folders, output_folder):
        self.input_folders = input_folders
        self.output_folder = output_folder
        self.create_output_directories()
        self.processed_count = 0
        self.total_images = 0

    def create_output_directories(self):
        """Create output directories for each input folder"""
        os.makedirs(self.output_folder, exist_ok=True)
        for folder in self.input_folders:
            folder_name = os.path.basename(folder)
            output_subfolder = os.path.join(self.output_folder, folder_name)
            os.makedirs(output_subfolder, exist_ok=True)

    def count_total_images(self):
        """Count total images across all folders"""
        image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}
        total = 0
        for folder in self.input_folders:
            if os.path.exists(folder):
                for root, dirs, files in os.walk(folder):
                    total += sum(1 for f in files if os.path.splitext(f.lower())[1] in image_extensions)
        return total

    def rotate_90_degrees(self, image):
        """Rotate image by 90, 180, or 270 degrees"""
        angles = [90, 180, 270]
        angle = random.choice(angles)
        return image.rotate(angle, expand=True)

    def width_shift(self, image, shift_range=0.3):
        """Shift image width by up to 30%"""
        width, height = image.size
        shift_pixels = int(width * shift_range * random.uniform(-1, 1))

        new_image = Image.new('RGB', (width, height), (255, 255, 255))
        if shift_pixels > 0:
            new_image.paste(image, (shift_pixels, 0))
        else:
            cropped = image.crop((-shift_pixels, 0, width, height))
            new_image.paste(cropped, (0, 0))

        return new_image

    def height_shift(self, image, shift_range=0.3):
        """Shift image height by up to 30%"""
        width, height = image.size
        shift_pixels = int(height * shift_range * random.uniform(-1, 1))

        new_image = Image.new('RGB', (width, height), (255, 255, 255))
        if shift_pixels > 0:
            new_image.paste(image, (0, shift_pixels))
        else:
            cropped = image.crop((0, -shift_pixels, width, height))
            new_image.paste(cropped, (0, 0))

        return new_image

    def crop_image(self, image, crop_percentage=0.15):
        """Crop image by 15%"""
        width, height = image.size
        crop_w = int(width * crop_percentage)
        crop_h = int(height * crop_percentage)

        left = random.randint(0, crop_w)
        top = random.randint(0, crop_h)
        right = width - random.randint(0, crop_w)
        bottom = height - random.randint(0, crop_h)

        cropped = image.crop((left, top, right, bottom))
        return cropped.resize((width, height), Image.Resampling.LANCZOS)

    def scale_image(self, image, scale_range=0.05):
        """Enlarge or reduce image by up to 5%"""
        width, height = image.size
        scale_factor = 1 + random.uniform(-scale_range, scale_range)

        new_width = int(width * scale_factor)
        new_height = int(height * scale_factor)

        scaled = image.resize((new_width, new_height), Image.Resampling.LANCZOS)

        if scale_factor > 1:
            left = (new_width - width) // 2
            top = (new_height - height) // 2
            scaled = scaled.crop((left, top, left + width, top + height))
        else:
            new_image = Image.new('RGB', (width, height), (255, 255, 255))
            paste_x = (width - new_width) // 2
            paste_y = (height - new_height) // 2
            new_image.paste(scaled, (paste_x, paste_y))
            scaled = new_image

        return scaled

    def color_shift(self, image):
        """Apply slight color changes"""
        hsv = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2HSV)

        hue_shift = random.randint(-10, 10)
        hsv[:, :, 0] = (hsv[:, :, 0] + hue_shift) % 180

        saturation_factor = random.uniform(0.9, 1.1)
        hsv[:, :, 1] = np.clip(hsv[:, :, 1] * saturation_factor, 0, 255)

        rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)
        return Image.fromarray(rgb)

    def flip_image(self, image):
        """Apply horizontal and/or vertical flips"""
        flip_type = random.choice(['horizontal', 'vertical', 'both'])

        if flip_type == 'horizontal':
            return image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
        elif flip_type == 'vertical':
            return image.transpose(Image.Transpose.FLIP_TOP_BOTTOM)
        else:
            flipped = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
            return flipped.transpose(Image.Transpose.FLIP_TOP_BOTTOM)

    def brightness_adjustment(self, image, brightness_range=0.8):
        """Adjust brightness by up to 80%"""
        enhancer = ImageEnhance.Brightness(image)
        factor = random.uniform(1 - brightness_range, 1 + brightness_range)
        return enhancer.enhance(factor)

    def apply_random_augmentation(self, image):
        augmentations = [
            self.rotate_90_degrees,
            self.width_shift,
            self.height_shift,
            self.crop_image,
            self.scale_image,
            self.color_shift,
            self.flip_image,
            self.brightness_adjustment
        ]

        augmentation = random.choice(augmentations)
        return augmentation(image)

    def show_progress(self):
        """Display progress bar"""
        if self.total_images > 0:
            progress = (self.processed_count / self.total_images) * 100
            bar_length = 50
            filled_length = int(bar_length * progress / 100)
            bar = '█' * filled_length + '-' * (bar_length - filled_length)
            print(f'\rProgress: |{bar}| {progress:.1f}% ({self.processed_count}/{self.total_images})', end='', flush=True)

    def process_folder(self, input_folder, num_augmentations=5):
        """Process all images in a folder and create augmented versions"""
        folder_name = os.path.basename(input_folder)
        output_subfolder = os.path.join(self.output_folder, folder_name)

        image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}
        image_files = []

        # Walk through all subdirectories
        for root, dirs, files in os.walk(input_folder):
            for file in files:
                if os.path.splitext(file.lower())[1] in image_extensions:
                    image_files.append(os.path.join(root, file))

        print(f"\nProcessing {len(image_files)} images in {folder_name}...")

        for img_file in image_files:
            try:
                image = Image.open(img_file).convert('RGB')

                base_name = os.path.splitext(os.path.basename(img_file))[0]
                original_output = os.path.join(output_subfolder, f"{base_name}_original.jpg")
                image.save(original_output, 'JPEG', quality=95)

                for i in range(num_augmentations):
                    augmented_image = self.apply_random_augmentation(image.copy())
                    augmented_filename = f"{base_name}_aug_{i+1}.jpg"
                    augmented_path = os.path.join(output_subfolder, augmented_filename)
                    augmented_image.save(augmented_path, 'JPEG', quality=95)

                self.processed_count += 1
                self.show_progress()

            except Exception as e:
                print(f"\nError processing {os.path.basename(img_file)}: {str(e)}")

    def augment_all_folders(self, num_augmentations=5):
        """Process all input folders"""
        self.total_images = self.count_total_images()
        print(f"\nStarting augmentation for {len(self.input_folders)} folders...")
        print(f"Total images to process: {self.total_images}")
        print(f"Creating {num_augmentations} augmented versions per image")
        print("-" * 70)

        for folder in self.input_folders:
            if os.path.exists(folder):
                self.process_folder(folder, num_augmentations)
            else:
                print(f"\nWarning: Folder {folder} does not exist")

        print(f"\n\nAugmentation completed!")
        print(f"Total images processed: {self.processed_count}")
        print(f"Total augmented images created: {self.processed_count * (num_augmentations + 1)}")

In [ ]:
output_folder = "/content/augmented_jewelry_images"
num_augmentations = 8  # Number of augmented versions per original image

# Create augmentor and process all folders
if input_folders:
    augmentor = JewelryImageAugmentor(input_folders, output_folder)
    augmentor.augment_all_folders(num_augmentations)
else:
    print("No input folders found. Please upload ZIP files containing your jewelry images.")

In [ ]:
def show_sample_results(output_folder, num_samples=3):
    """Display sample original and augmented images"""
    sample_count = 0

    for folder_name in os.listdir(output_folder):
        folder_path = os.path.join(output_folder, folder_name)
        if os.path.isdir(folder_path):
            print(f"\n=== {folder_name.upper()} SAMPLES ===")

            # Get original images
            original_files = [f for f in os.listdir(folder_path) if 'original' in f]

            for orig_file in original_files[:num_samples]:
                base_name = orig_file.replace('_original.jpg', '')

                # Find corresponding augmented images
                aug_files = [f for f in os.listdir(folder_path) if f.startswith(base_name + '_aug_')]

                if aug_files:
                    # Display original and first few augmented versions
                    fig, axes = plt.subplots(1, min(5, len(aug_files) + 1), figsize=(15, 3))
                    if len(aug_files) == 0:
                        axes = [axes]

                    # Show original
                    orig_path = os.path.join(folder_path, orig_file)
                    orig_img = Image.open(orig_path)
                    axes[0].imshow(orig_img)
                    axes[0].set_title('Original')
                    axes[0].axis('off')

                    # Show augmented versions
                    for i, aug_file in enumerate(sorted(aug_files)[:4]):
                        aug_path = os.path.join(folder_path, aug_file)
                        aug_img = Image.open(aug_path)
                        axes[i+1].imshow(aug_img)
                        axes[i+1].set_title(f'Augmented {i+1}')
                        axes[i+1].axis('off')

                    plt.tight_layout()
                    plt.show()

                    sample_count += 1
                    if sample_count >= num_samples:
                        break

# Show sample results
if os.path.exists(output_folder):
    show_sample_results(output_folder, num_samples=2)

In [ ]:
def create_download_zip():
    """Create a ZIP file of all augmented images for download"""
    zip_filename = "/content/augmented_jewelry_images.zip"

    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(output_folder):
            for file in files:
                file_path = os.path.join(root, file)
                arc_name = os.path.relpath(file_path, "/content/")
                zipf.write(file_path, arc_name)

    print(f"Created download ZIP: {zip_filename}")
    print(f"ZIP file size: {os.path.getsize(zip_filename) / (1024*1024):.2f} MB")

    return zip_filename

# Create and download the results
if os.path.exists(output_folder):
    zip_path = create_download_zip()

    # Download the ZIP file
    print("\nDownloading augmented images...")
    files.download(zip_path)

    # Optional: Save to Google Drive (uncomment if Drive is mounted)
    # drive_path = "/content/drive/MyDrive/augmented_jewelry_images.zip"
    # shutil.copy2(zip_path, drive_path)
    # print(f"Also saved to Google Drive: {drive_path}")

In [ ]:
def print_summary_stats():
    """Print summary of augmentation results"""
    if not os.path.exists(output_folder):
        print("No output folder found.")
        return

    total_images = 0
    folder_stats = {}

    for folder_name in os.listdir(output_folder):
        folder_path = os.path.join(output_folder, folder_name)
        if os.path.isdir(folder_path):
            folder_images = len([f for f in os.listdir(folder_path) if f.endswith('.jpg')])
            folder_stats[folder_name] = folder_images
            total_images += folder_images

    print("\n" + "="*50)
    print("AUGMENTATION SUMMARY")
    print("="*50)

    for folder_name, count in folder_stats.items():
        original_count = len([f for f in os.listdir(os.path.join(output_folder, folder_name)) if 'original' in f])
        augmented_count = count - original_count
        print(f"{folder_name}: {original_count} original → {count} total ({augmented_count} augmented)")

    print("-"*50)
    print(f"TOTAL: {total_images} images created")
    print("="*50)

print_summary_stats()